# Step 2-2. Multi-Agent: 전문가 세 명과 지휘자
블로그의 구조를 그대로 만듭니다.

```
질문 ─▶ 지휘자(Orchestrator) ─┬─▶ 의약화학 전문가   (PubChem, RDKit)
                              ├─▶ 임상약리 전문가   (PubMed, openFDA)
                              └─▶ 임상·규제 전문가  (ClinicalTrials.gov)
                                        │
                              지휘자가 세 답을 모아 통합 리포트
```

## 두 가지 기능
1. **종합 분석**: 세 명이 모두 분석하고 Go / No-Go 리포트를 씁니다
2. **스마트 컨설팅**: 질문을 보고 **필요한 전문가만** 부릅니다. 블로그의 "AI CEO"에 해당합니다

## 핵심 개념
- **Orchestrator(지휘자)**: 일을 쪼개고, 맡기고, 모으는 Agent
- **Sub-agent**: 저마다 자기 역할과 도구만 가진 전문가. 서로의 대화 내용은 모릅니다
- **Routing(라우팅)**: 질문을 어느 전문가에게 보낼지 정하는 일

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1~2분 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "strands-agents[ollama]", "rdkit", "requests", "ollama"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정

In [ ]:
import os, sys
import ollama

sys.path.insert(0, os.path.abspath(".."))   # 저장소 맨 위의 drug_tools.py를 불러오기 위해

EXPERT_MODEL = "gemma4:e4b"    # 전문가와 지휘자 역할을 하는 모델 (약 9.5GB)
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")   # GPU 서버를 쓰면 그 주소로 바뀝니다

installed = [m.model for m in ollama.list().models]
if not any(m == EXPERT_MODEL or m.startswith(EXPERT_MODEL + ":") for m in installed):
    print(f"⬇️  {EXPERT_MODEL} 내려받는 중... (약 9.5GB, 처음 한 번만)")
    ollama.pull(EXPERT_MODEL)
print(f"✅ 모델 준비 완료: {EXPERT_MODEL}  (Ollama 주소: {OLLAMA_HOST})")

## 2. 전문가 세 명 정의
Step 2-1의 전문가를 세 명으로 늘립니다. 전문가마다 **역할 설명과 도구가 다릅니다.** 임상약리 전문가는 화합물 구조 도구를 못 쓰고, 화학 전문가는 임상시험을 못 찾습니다.

In [ ]:
import json, time
from concurrent.futures import ThreadPoolExecutor
import drug_tools as dt
from strands import Agent, tool
from strands.models.ollama import OllamaModel


@tool
def compound_info(name: str) -> dict:
    """화합물의 영문 이름으로 PubChem에서 기본 정보와 SMILES 구조식을 찾는다."""
    return dt.compound_info(name)


@tool
def molecule_properties(smiles: str) -> dict:
    """SMILES 구조식으로 분자량, 지용성(logP), 수소결합 수, Lipinski 규칙 통과 여부를 계산한다."""
    return dt.molecule_properties(smiles)


@tool
def similar_compounds(smiles: str) -> dict:
    """SMILES 구조식과 구조가 비슷한 화합물을 PubChem에서 찾는다."""
    return dt.similar_compounds(smiles, 5)


@tool
def pubmed_search(query: str) -> dict:
    """PubMed에서 논문을 검색한다. 검색어는 영문으로 쓴다 (예: "ibuprofen cardiotoxicity")."""
    return dt.pubmed_search(query, 5)


@tool
def adverse_events(drug: str) -> dict:
    """openFDA에 가장 많이 보고된 부작용을 찾는다. 약의 영문 성분명을 넣는다."""
    return dt.adverse_events(drug, 8)


@tool
def clinical_trials(query: str) -> dict:
    """ClinicalTrials.gov에서 임상시험을 검색한다. 약 이름이나 질환을 영문으로 넣는다."""
    return dt.clinical_trials(query, 5)


COMMON = """
- 반드시 도구를 써서 근거를 찾는다. 도구 결과에 없는 숫자와 사실은 쓰지 않는다.
- 한국어로, 소제목 3개 이내, 전체 10줄 이내로 답한다.
- 마지막 줄에 확신 정도(높음, 보통, 낮음)를 적는다."""

EXPERTS = {
    "chemist": {
        "title": "의약화학 전문가",
        "prompt": "너는 15년 경력의 의약화학 전문가다. 화합물의 구조, 물리화학적 특성, 구조가 비슷한 화합물을 보고 구조를 어떻게 고치면 좋을지 판단한다." + COMMON,
        "tools": [compound_info, molecule_properties, similar_compounds],
    },
    "pharmacologist": {
        "title": "임상약리 전문가",
        "prompt": "너는 20년 경력의 임상약리 전문가다. 논문과 부작용 보고를 근거로 안전성, 독성, 약물 상호작용, 용량을 판단한다." + COMMON,
        "tools": [pubmed_search, adverse_events],
    },
    "regulatory": {
        "title": "임상·규제 전략 전문가",
        "prompt": "너는 신약 허가 전략 전문가다. 비슷한 약의 임상시험 현황을 근거로 임상 설계, 허가 경로, 경쟁 상황을 판단한다." + COMMON,
        "tools": [clinical_trials, pubmed_search],
    },
}


def make_model():
    return OllamaModel(host=OLLAMA_HOST, model_id=EXPERT_MODEL, temperature=0.2)


def ask_expert(key, question, context):
    """전문가 한 명에게 묻는다. 매번 새 Agent를 만들어 다른 질문의 기억이 섞이지 않게 한다."""
    spec = EXPERTS[key]
    agent = Agent(model=make_model(), system_prompt=spec["prompt"], tools=spec["tools"], callback_handler=None)
    start = time.time()
    used = lambda: [b["toolUse"]["name"] for m in agent.messages for b in m.get("content", []) if "toolUse" in b]
    try:
        text = str(agent(f"[화합물 기본 정보]\n{context}\n\n[질문]\n{question}")).strip()
        if not used():      # 도구 없이 기억만으로 답했으면 한 번 더 시킨다
            text = str(agent("도구를 한 번도 쓰지 않았다. 먼저 도구를 불러 근거를 찾고, 그 결과만으로 다시 답하라.")).strip()
    except Exception as e:
        text = f"(분석 실패: {e})"
    tools = used()
    return {"expert": key, "title": spec["title"], "answer": text, "tools": tools, "seconds": round(time.time() - start, 1)}


print("✅ 전문가 3명 준비 완료:", ", ".join(v["title"] for v in EXPERTS.values()))

## 3. 공통 기초 자료 만들기
블로그의 1단계(분자 특징 예측)에 해당합니다. 화합물 정보와 계산한 특성을 한 번 만들어 **모든 전문가에게 똑같이 줍니다.**

In [ ]:
def base_context(drug):
    info = dt.compound_info(drug)
    if not info.get("found"):
        raise SystemExit(f"❌ PubChem에서 '{drug}'를 찾지 못했습니다. 영문 이름을 확인하세요.")
    props = dt.molecule_properties(info["smiles"])
    return (f"이름: {drug}\nSMILES: {info['smiles']}\n분자식: {info['formula']}\n"
            f"분자량: {props['molecular_weight']}, logP: {props['logp']}, 수소결합 주개/받개: {props['h_bond_donors']}/{props['h_bond_acceptors']}, "
            f"TPSA: {props['tpsa']}, QED: {props['qed']}, Lipinski 통과: {props['lipinski_pass']}")


# TODO: 다른 약 이름(영문)으로 바꿔 보세요!
DRUG = "ibuprofen"
CONTEXT = base_context(DRUG)
print(CONTEXT)

## 4. 기능 1: 종합 분석
세 전문가에게 **동시에** 일을 맡기고, 지휘자가 결과를 모아 리포트를 씁니다. 전문가 한 명에 20\~40초, 전체 1\~2분쯤 걸립니다.

In [ ]:
def run_experts(keys, question, context):
    """고른 전문가들에게 동시에 일을 맡긴다."""
    with ThreadPoolExecutor(max_workers=len(keys)) as pool:
        return list(pool.map(lambda k: ask_expert(k, question, context), keys))


def synthesize(question, context, results):
    """지휘자가 전문가들의 답을 모아 하나의 리포트로 쓴다."""
    parts = "\n\n".join(f"### {r['title']}의 분석\n{r['answer']}" for r in results)
    prompt = f"""너는 신약 개발팀의 지휘자다. 아래 전문가 분석만 근거로 한국어 통합 리포트를 쓴다.
형식: (1) 한 줄 결론 (2) 근거 3개 이내 (3) 위험 요소 (4) 다음에 할 일. 전문가 분석에 없는 내용은 쓰지 않는다. 전체 12줄 이내.

[화합물 기본 정보]
{context}

[질문]
{question}

{parts}"""
    r = ollama.chat(model=EXPERT_MODEL, messages=[{"role": "user", "content": prompt}], options={"temperature": 0.2})
    return r["message"]["content"].strip()


def show(results):
    for r in results:
        print(f"\n🧑‍🔬 {r['title']}  ({r['seconds']}초, 도구: {' → '.join(r['tools']) or '안 씀'})")
        print(r["answer"])


question = f"{DRUG}를 다음 개발 단계로 진행해도 되는지 종합 평가해 줘"
start = time.time()
results = run_experts(list(EXPERTS), question, CONTEXT)
show(results)
report = synthesize(question, CONTEXT, results)
print("\n" + "=" * 60 + "\n📋 통합 리포트\n" + "=" * 60)
print(report)
print(f"\n⏱️ 전체 {time.time() - start:.0f}초")

## 5. 기능 2: 스마트 컨설팅 (필요한 전문가만 부르기)
모든 질문에 세 명을 다 부르면 느리고 낭비입니다. 지휘자가 먼저 **누가 필요한지** 고릅니다. 답을 정해진 형식(JSON)으로만 받게 해서, 코드가 그 결과를 그대로 쓸 수 있게 합니다.

In [ ]:
ROUTER_PROMPT = """너는 신약 개발팀의 지휘자다. 질문에 답하는 데 꼭 필요한 전문가만 고른다.
- chemist: 화합물 구조, 물리화학적 특성, 구조 변경, 유사 화합물
- pharmacologist: 약동학, 약력학, 독성, 부작용, 약물 상호작용, 용량
- regulatory: 임상시험 현황과 설계, 허가 전략, 시장과 경쟁
질문 하나에 1명에서 3명까지 고를 수 있다."""
ROUTER_SCHEMA = {"type": "object", "required": ["experts"], "properties": {"experts": {
    "type": "array", "minItems": 1, "maxItems": 3,
    "items": {"type": "string", "enum": ["chemist", "pharmacologist", "regulatory"]}}}}
ORDER = ["chemist", "pharmacologist", "regulatory"]


def route_with_teacher(question):
    """큰 모델(지휘자)이 필요한 전문가를 고른다. 답은 정해진 형식(JSON)으로만 받는다."""
    kw = dict(model=EXPERT_MODEL, format=ROUTER_SCHEMA, options={"temperature": 0},
              messages=[{"role": "system", "content": ROUTER_PROMPT}, {"role": "user", "content": question}])
    try:
        r = ollama.chat(think=False, **kw)     # 생각 과정을 끄면 빨라집니다
    except Exception:
        r = ollama.chat(**kw)
    return sorted(set(json.loads(r["message"]["content"])["experts"]), key=ORDER.index)


for q in ["이 화합물의 심장독성 위험은?", "FDA 승인 가능성은?", "용해도를 높이려면 구조를 어떻게 바꿔야 해?"]:
    start = time.time()
    print(f"{q}  →  {route_with_teacher(q)}  ({time.time() - start:.1f}초)")

In [ ]:
def consult(question, router=route_with_teacher):
    """질문 하나를 처음부터 끝까지 처리한다: 전문가 고르기 → 맡기기 → 모으기."""
    start = time.time()
    keys = router(question)
    print(f"👤 질문: {question}")
    print(f"🧭 지휘자의 선택: {', '.join(EXPERTS[k]['title'] for k in keys)}")
    results = run_experts(keys, question, CONTEXT)
    show(results)
    answer = synthesize(question, CONTEXT, results) if len(results) > 1 else results[0]["answer"]
    print("\n📋 통합 답변\n" + answer)
    print(f"\n⏱️ {time.time() - start:.0f}초, 전문가 {len(keys)}명")
    return answer


# TODO: 질문을 바꿔 보세요!
_ = consult(f"{DRUG}를 신장이 나쁜 환자에게 써도 될까?")

### 직접 해 보기
질문에 따라 부르는 전문가 수와 걸리는 시간이 달라지는지 보세요.

In [ ]:
# TODO: 질문을 바꿔 보세요!
_ = consult(f"{DRUG}와 비슷한 약의 임상시험 현황은?")

## 6. 리포트를 파일로 저장

In [ ]:
os.makedirs("reports", exist_ok=True)
path = f"reports/{DRUG}_report.md"
with open(path, "w", encoding="utf-8") as f:
    f.write(f"# {DRUG} 검토 리포트\n\n## 기본 정보\n```\n{CONTEXT}\n```\n\n## 통합 리포트\n{report}\n")
    for r in results:
        f.write(f"\n## {r['title']}\n도구: {', '.join(r['tools'])}\n\n{r['answer']}\n")
    f.write("\n> 교육·연구용 실습 결과입니다. 임상 판단을 대신하지 않습니다.\n")
print(f"✅ 저장했습니다: example/{path}")

## 정리
- Multi-Agent는 "전문가 여러 명 + 지휘자"입니다. 전문가는 자기 도구만 쓰고, 지휘자는 고르고 모읍니다.
- **지휘자의 일 가운데 "누구에게 보낼지 고르기"는 아주 단순합니다.** 그런데 여기에도 9.5GB짜리 큰 모델을 쓰고 있습니다.
- 이 단순한 일을 훨씬 작은 모델에게 가르칠 수 있을까요? 그것이 Distillation(증류)입니다. Step 2-4에서 해 봅니다.
- 다음은 내 주제로 팀을 만들어 서비스로 띄웁니다. → Step 2-3